# Day 4 v2 — Model 14: AITeamVN Phased Gradual Unfreezing + Price-Bin Aux Head

**Architecture:** `AITeamVN/Vietnamese_Embedding` (568M, 24L) with two combined improvements:

1. **Phased gradual unfreezing** — encoder adapts in 3 phases:
   - Phase 1 (ep 1–4): top-4 layers, lr=2e-5  — gentle adaptation
   - Phase 2 (ep 5–8): top-8 layers, lr=1e-5  — deeper features
   - Phase 3 (ep 9–12): top-16 layers, lr=5e-6 — near-full encoder

2. **Price-Bin aux head** — 5 log-spaced price bins instead of coarse 8-class categories:
   - Bin 0: price < 50k  |  Bin 1: 50–150k  |  Bin 2: 150–350k  |  Bin 3: 350–650k  |  Bin 4: ≥650k
   - Direct price range signal → stronger gradient than category classification

| Config | Value |
|---|---|
| batch | 24 |
| base_lr (Phase 1→2→3) | 2e-5 → 1e-5 → 5e-6 |
| llrd_decay | 0.85 |
| aux_alpha | 0.15 (vs 0.1 for category) |
| R-Drop | 0.3 |
| EMA | 0.9999 |
| epochs | 12 (3×4) |
| patience | 5 (cross-phase) |

**Target:** MAE < 65k VND (MOST PROMISING architecture)

## vast.ai Setup (chỉ chạy lần đầu)

```bash
pip install uv
uv sync
```

Restart kernel sau khi sync xong.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import json
import torch

from pricer_vi_2.items import Item
from pricer_vi_2.evaluator import evaluate, plot_training_history
from pricer_vi_2.bert_finetune_phased_model import PhasedBERTRunner

MODEL_NAME   = "AITeamVN/Vietnamese_Embedding"
WEIGHT_DIR   = Path("weights")
VAL_PRED_DIR = Path("val_predictions")

# A14 config — phased unfreezing + price-bin aux head
BATCH        = 24
BASE_LR      = 2e-5    # Phase 1 lr; Phase 2=1e-5, Phase 3=5e-6 (set inside runner)
WEIGHT_DECAY = 0.01
LLRD_DECAY   = 0.85
EPOCHS       = 12      # 3 phases x 4 epochs
PATIENCE     = 5       # cross-phase patience
EMA_DECAY    = 0.9999
WARMUP_RATIO = 0.05    # warmup per phase
R_DROP_ALPHA = 0.3
AUX_ALPHA    = 0.15    # price-bin signal stronger than category

print(f"torch: {torch.__version__} | cuda: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Load Data

In [ ]:
train, val, test = Item.from_hub("SeanSunny/items_tv_v9")
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

## 2. Setup Runner

- Tokenize 269K train + 3926 val (~550MB at max_length=256)
- Compute price bin labels: 5 log-spaced bins [<50k, 50-150k, 150-350k, 350-650k, ≥650k]
- Phase 1 start: top-4 layers unfrozen, ~16M trainable params

In [ ]:
runner = PhasedBERTRunner(train, val)

runner.setup(
    model_name=MODEL_NAME,
    batch_size=BATCH,
    max_length=256,
    base_lr=BASE_LR,
    weight_decay=WEIGHT_DECAY,
    llrd_decay=LLRD_DECAY,
    dropout=0.2,
)

## 3. Train

12 epochs across 3 phases. Optimizer + scheduler rebuilt at each phase boundary.
Patience=5 (cross-phase: model may dip at phase transitions before improving).
Expected: ~50-60 min/epoch → ~10-12h total on RTX 3090 Ti.

In [ ]:
history = runner.train(
    epochs=EPOCHS,
    patience=PATIENCE,
    huber_delta=1.0,
    aux_alpha=AUX_ALPHA,
    ema_decay=EMA_DECAY,
    warmup_ratio=WARMUP_RATIO,
    max_grad_norm=1.0,
    r_drop_alpha=R_DROP_ALPHA,
)

## 4. Training History

In [ ]:
plot_training_history(history, title="AITeamVN Phased Unfreeze + Price-Bin Aux Head")

## 5. Save Weights + Val Predictions + Test Predictions

In [ ]:
WEIGHT_DIR.mkdir(exist_ok=True)
runner.save(str(WEIGHT_DIR / "aitvn_phased.pth"))
print("Saved weights/aitvn_phased.pth")

VAL_PRED_DIR.mkdir(exist_ok=True)

print("Running val predictions (3926 samples)...")
val_preds = runner.val_predictions()
with open(VAL_PRED_DIR / "aitvn_phased_val.json", "w") as f:
    json.dump(val_preds, f)
print(f"Saved val_predictions/aitvn_phased_val.json ({len(val_preds)} samples)")

print("Running test predictions (3872 samples)...")
test_preds = runner.test_predictions(test)
with open(VAL_PRED_DIR / "aitvn_phased_test.json", "w") as f:
    json.dump(test_preds, f)
print(f"Saved val_predictions/aitvn_phased_test.json ({len(test_preds)} samples)")

## 6. Evaluate on 200 Test Samples

In [ ]:
def aitvn_phased_pricer(item):
    return runner.inference(item)

results = evaluate(aitvn_phased_pricer, test)
print(f"MAE: {results['mae']:.1f}k VND | MSE: {results['mse']:,.0f} | R2: {results['r2']:.1f}%")

## 7. Sanity Check

In [ ]:
sample = test[0]
pred = runner.inference(sample)
print(f"Product: {sample.title[:60]}")
print(f"Actual:  {sample.price:.1f}k VND")
print(f"Predict: {pred:.1f}k VND")
print(f"Error:   {abs(pred - sample.price):.1f}k VND")

ckpt = torch.load(str(WEIGHT_DIR / "aitvn_phased.pth"), map_location="cpu", weights_only=False)
print(f"\nCheckpoint keys: {sorted(ckpt.keys())}")
print(f"model_name={ckpt['model_name']}")
print(f"y_mean={ckpt['y_mean']:.4f} | y_std={ckpt['y_std']:.4f}")
print(f"price_bin_boundaries={ckpt['price_bin_boundaries']}")